# 016 — XGBoost 007 + MLP 015

## Objetivo e protocolo

Repete a mistura do 014, com 75% das probabilidades do XGBoost 007 e 25% do MLP 015. O XGBoost mantém `multi:softprob`, `hist`, 200 árvores, profundidade 5, taxa 0.1 e semente 42; o MLP usa arquitetura 64 → 32, ReLU, Adam, `weight_decay=0`, `batch_size=128`, taxa 0.0001 e 93 épocas. As probabilidades do MLP são obtidas com softmax e ambas as saídas são alinhadas às classes 0–15.

A divisão 80/20 e a validação interna de 15% repetem exatamente o protocolo estratificado do 015 (`random_state=42`). Cada preparação é ajustada só nas linhas de treino correspondentes. A configuração 0.0001/93 é a selecionada no 015; não é reajustada na validação fixa. Após a análise interna, treinamos modelos novos nos 80%, avaliamos uma vez nos 20% fixos e treinamos/exportamos outros modelos nas 10.962 linhas.

Referência: 014 Macro F1 fixa **0.4613**, Kaggle **0.47**; MLP 015 substitui o MLP 012 do 014.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

repo_root = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents]
                 if (p / "src" / "experiments_016_018.py").is_file())
sys.path.insert(0, str(repo_root))
from src.experiments_016_018 import run_016

result = run_016()
metrics = result["metrics"]
comparison = pd.DataFrame([
    {"modelo": "XGBoost 007", **result["xgboost_metrics"]},
    {"modelo": "MLP 015 (0.0001, 93 épocas)", **result["mlp015_metrics"]},
    {"modelo": "Mistura 75/25", **metrics},
])[["modelo", "macro_f1", "accuracy"]]
comparison["diferença vs 014"] = [
    result["xgboost_metrics"]["macro_f1"] - 0.4613,
    result["mlp015_metrics"]["macro_f1"] - 0.4613,
    metrics["macro_f1"] - 0.4613,
]
display(comparison.style.format({"macro_f1": "{:.4f}", "accuracy": "{:.4f}", "diferença vs 014": "{:+.4f}"}))
class_table = pd.DataFrame(metrics["per_class"])
display(class_table.style.format({"F1": "{:.4f}", "recall": "{:.4f}"}).apply(
    lambda row: ["font-weight: bold" if row.classe in [1, 6, 13] else "" for _ in row], axis=1
))
print(f"Macro F1 interna da mistura: {result['inner_metrics']['macro_f1']:.4f}")
print(f"Efeito da troca face ao 014: {metrics['macro_f1'] - 0.4613:+.4f} Macro F1 fixa")
print(f"Exploratória: {result['exploratory']}; CSV: {result['submission']}")
print("Treino MLP em segundos:", result["training_seconds"])

,modelo,macro_f1,accuracy,diferença vs 014
0,XGBoost 007,0.4491,0.4583,-0.0122
1,"MLP 015 (0.0001, 93 épocas)",0.4224,0.4191,-0.0389
2,Mistura 75/25,0.4580,0.4587,-0.0033


,classe,escalão,F1,recall,support
0,0,15–17 anos,0.8140,0.8537,41
1,1,18–19 anos,0.4286,0.3529,34
2,2,20–24 anos,0.5890,0.6000,80
3,3,25–29 anos,0.3624,0.3600,75
4,4,30–34 anos,0.2609,0.2333,90
5,5,35–39 anos,0.3333,0.3488,129
6,6,40–44 anos,0.3796,0.3895,172
7,7,45–49 anos,0.3780,0.3584,173
8,8,50–54 anos,0.6094,0.6464,181
9,9,55–59 anos,0.6135,0.6256,203


Macro F1 interna da mistura: 0.4859
Efeito da troca face ao 014: -0.0033 Macro F1 fixa
Exploratória: True; CSV: submissions/submission_016_xgboost_mlp015.csv
Treino MLP em segundos: {'mlp_inner': 19.37945262499852, 'mlp_80_percent': 21.88865670800442, 'mlp_export': 26.49628004198894}


## Conclusão

- Data: 11/11; pontuação Kaggle: 0.48, o melhor resultado Kaggle atual.
- Macro F1 fixa: 0.4580, menos 0.0033 que o 014 (0.4613); a validação fixa já consultada é apenas comparação de desenvolvimento.
- O Kaggle supera os 0.46 do 017 e os 0.44 do 018.
- Melhorias: afinar pesos perto de 75/25.
